In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))   # temporary import shim; proper packaging later

import pandas as pd
from prospect.geology import get_geology

DATA = Path("..") / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

In [2]:
df = pd.read_csv(RAW / "mrds-GA.csv", low_memory=False, encoding="latin-1")
keep = ["dep_id", "site_name", "latitude", "longitude", "commod1"]
df = df[keep].dropna(subset=["latitude", "longitude"])
print(df.shape)

(2656, 5)


In [3]:
import time

sample = df.sample(25, random_state=42).reset_index(drop=True)

rows = []
for i, rec in sample.iterrows():
    geo = get_geology(rec["latitude"], rec["longitude"])
    rows.append({
        "dep_id": rec["dep_id"],
        "site_name": rec["site_name"],
        "commodity": rec["commod1"],
        "lat": rec["latitude"],
        "lng": rec["longitude"],
        **geo,
    })
    print(f"{i+1:>2}/25  {str(rec['site_name'])[:35]:<35} -> "
          f"[{geo['status']}] {geo.get('unit_name', '')}")
    time.sleep(0.5)

table = pd.DataFrame(rows)
table.to_csv(PROCESSED / "sample_25_hardened.csv", index=False)
table["status"].value_counts()

 1/25  Kelly O'Neil Prospects              -> [OK] Paleozoic sedimentary rocks
 2/25  Penland Hill                        -> [OK] Knox Group undifferentiated
 3/25  H.E.Lucas Property                  -> [OK] Knox Group undifferentiated
 4/25  Crosby No. 3 Mine                   -> [OK] Late Cretaceous sedimentary
 5/25  Stovall Pit                         -> [OK] Neoproterozoic-Cambrian volcanic: interlayered sedimentary and volcanic rocks
 6/25  Tallapoosa Mine                     -> [OK] Paleozoic sedimentary rocks
 7/25  Prater Mine                         -> [OK] Neoproterozoic-Cambrian sedimentary
 8/25  Huber Mill                          -> [OK] Twiggs Clay
 9/25  Young Deer Creek                    -> [OK] Mesoproterozoic crystalline metamorphic rocks
10/25  Lot 826                             -> [OK] Rome Formation
11/25  Clay Preparation Plant              -> [OK] Late Eocene sedimentary
12/25  Lots 191 and 242                    -> [OK] Chilhowee Formation
13/25  Draketown 

status
OK    25
Name: count, dtype: int64